# Config and Persistence

In production, decision models need to evolve without code deploys. Decider stores every module as a versioned JSON config, so you can update weights, thresholds, and logic by writing a new config version and pointing your server at it — zero downtime, full audit trail.

In [ ]:
%pip install -q decider


In [ ]:
import warnings
warnings.filterwarnings('ignore')


import polars as pl
from pydantic import BaseModel
from decider.modules.functional import generate_from_functions
from decider.config.file import JsonFileConfigManager
from decider.modules import GraphModule

print('Imports OK')

## Build a Scorer with Specific Config

We will use a simple risk scorer whose weights are controlled by a Pydantic config.

In [ ]:
class RiskConfig(BaseModel):
    dti_weight:  float = 150.0
    base_score:  float = 700.0

def dti_ratio(debt: pl.Expr, income: pl.Expr) -> pl.Expr:
    return debt / income

def risk_score(dti_ratio: pl.Expr, config: RiskConfig) -> pl.Expr:
    return pl.lit(config.base_score) - dti_ratio * pl.lit(config.dti_weight)

RiskScorer = generate_from_functions('risk_scorer', dti_ratio, risk_score)

# Instantiate with specific weights
scorer = RiskScorer(name='risk_scorer', dti_weight=200.0, base_score=800.0)

df = pl.DataFrame({
    'applicant_id': ['A1', 'A2', 'A3'],
    'debt':         [20000.0, 5000.0,  50000.0],
    'income':       [40000.0, 60000.0, 80000.0],
})

result = scorer({'input': df})
print(result.select(['applicant_id', 'dti_ratio', 'risk_score']))

## Save to Versioned JSON Config

`asave` stages the module in a `VersionedConfig` object. `save_version` writes it to disk under `{basepath}/{version}/main.json`.

In [ ]:
# Use a temporary directory so this notebook is self-contained
CONFIGS_DIR = os.path.join(tempfile.mkdtemp(), 'configs')
print('Saving configs to:', CONFIGS_DIR)

mgr = JsonFileConfigManager(basepath=CONFIGS_DIR)

versioned = asyncio.run(scorer.asave('main', mgr))
asyncio.run(mgr.save_version(overwrite=True))

print('Saved version:', versioned.version)

## Load it Back

Create a fresh `JsonFileConfigManager` (simulating what a server does at startup), call `get_latest()`, and reconstruct the module with `GraphModule.model_validate`.

In [ ]:
fresh_mgr = JsonFileConfigManager(basepath=CONFIGS_DIR)
loaded = asyncio.run(fresh_mgr.get_latest())

print('Loaded version:', loaded.version)

module = GraphModule.model_validate(loaded.config['main']).root

print('Type          :', module.type)
print('dti_weight    :', module.dti_weight)
print('base_score    :', module.base_score)

# Verify the reconstructed module produces the same output
result2 = module({'input': df})
print(result2.select(['applicant_id', 'dti_ratio', 'risk_score']))

## Inspect the JSON on Disk

The config is plain JSON — human-readable, diff-friendly, and version-controllable.

In [ ]:
version_str = str(versioned.version)
json_path = os.path.join(CONFIGS_DIR, version_str, 'main.json')

with open(json_path) as f:
    on_disk = json.load(f)

print(json.dumps(on_disk, indent=2))

## Swap Config Values Without Redeploying

Save two versions with different weights, load both, and confirm their outputs differ. This is the core of Decider's hot-swap capability.

In [ ]:
CONFIGS_DIR_V2 = os.path.join(tempfile.mkdtemp(), 'configs_v2')

# --- Version 1: conservative weights ----------------------------------------
scorer_v1 = RiskScorer(name='risk_scorer', dti_weight=100.0, base_score=700.0)
mgr_v1 = JsonFileConfigManager(basepath=CONFIGS_DIR_V2)
versioned_1 = asyncio.run(scorer_v1.asave('main', mgr_v1))
asyncio.run(mgr_v1.save_version(overwrite=True))
print('Saved v1:', versioned_1.version, '  dti_weight=100  base_score=700')

# --- Version 2: aggressive weights ------------------------------------------
scorer_v2 = RiskScorer(name='risk_scorer', dti_weight=300.0, base_score=850.0)
mgr_v2 = JsonFileConfigManager(basepath=CONFIGS_DIR_V2)
ver2 = asyncio.run(scorer_v2.asave('main', mgr_v2))
asyncio.run(mgr_v2.save_version())
print('Saved v2:', ver2.version, '  dti_weight=300  base_score=850')

In [ ]:
# Load BOTH versions and compare outputs on the same data
all_versions = sorted(
    [d for d in os.listdir(CONFIGS_DIR_V2) if os.path.isdir(os.path.join(CONFIGS_DIR_V2, d))]
)
print('Versions on disk:', all_versions)

for ver_str in all_versions:
    r_mgr = JsonFileConfigManager(basepath=CONFIGS_DIR_V2)
    ver_cfg = asyncio.run(r_mgr._load_version(ver_str))
    mod = GraphModule.model_validate(ver_cfg.config['main']).root
    out = mod({'input': df}).select(['applicant_id', 'risk_score'])
    print('\nVersion', ver_str, ' dti_weight=' + str(mod.dti_weight) + '  base_score=' + str(mod.base_score))
    print(out)